# P4: Congressional disclosures, from source documents to searchable records
Sean Ryan | sryan3 | DSA 405 (002)
P3 supplies the acquisition, extraction and verified integration outline. P4 adds a real character repair, a documented portfolio snapshot, final report and reflection. The summary join is supporting technical evidence; it is not the House-only network.

## 0. Setup and independent inputs
Run from the repository or its `notebooks/` directory. Install `requirements.txt` in a Python 3.12 environment first.

**Colab:** This repository is private. Download the repository ZIP (or the supplied `P3-project.zip`) while signed in, then run this cell and upload that ZIP when prompted. The cell extracts the project safely and installs its declared dependencies in Colab. No GitHub token belongs in this notebook.

In [1]:
from pathlib import Path
import sys, json, hashlib, zipfile, subprocess, importlib.util

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "P3"]
ROOT = next((p for p in candidates if (p / "scripts/integration.py").is_file()), None)
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if ROOT is None and IN_COLAB:
    from google.colab import files
    uploads = files.upload()
    archives = [name for name in uploads if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload exactly one P4 project ZIP, containing scripts/ and data/raw/.")
    destination = Path("/content/p3_uploaded_project")
    destination.mkdir(exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()):
                raise ValueError("Unsafe ZIP member path")
        archive.extractall(destination)
    ROOT = next((p.parent.parent for p in destination.rglob("scripts/integration.py")), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete P4 repository, or upload its ZIP in Colab.")
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")])
sys.path.insert(0, str(ROOT / "scripts"))
import pandas as pd
try:
    from IPython.display import display, Code, HTML, SVG
except ImportError:
    from plain_display import display, Code, HTML, SVG
from integration import (verify_raw, load_house, prepare_comparable, aggregate,
                         join_design, merge_summaries, verify_join)
from senate import parse_archive, find_table, SourceStructureError
from acquisition import CachedFetcher, acquisition_plan, USER_AGENT
from build_site import failure_cases
print("Project root located. Python", sys.version.split()[0])
print("Unchanged original raw files verified:", verify_raw(ROOT))
display(pd.DataFrame(json.loads((ROOT / "provenance/source_commits.json").read_text())).T)


Project root located. Python 3.13.0
Unchanged original raw files verified: 1105
                                             commit
P2         429c32a23a9d3236f59df1a94a321fbc0054afb1
Senate     611e7ff2ffe5a26b9b5d2a6978cc4a5c85cf8d37
Portfolio  f93fbbebc6f281f70f76f8c61fc94dd942ffe70f


## 1. Acquisition code
The source inventory is fixed, with 515 archived House PDFs and 583 Senate HTML filings. A finite inventory has no pagination; all 1,098 document URLs are enumerated. The original Senate live-search pagination code is retained for provenance in `provenance/senate_scraper_original.py`; this version does not run that agreement-gated search.

`CachedFetcher` identifies the requests as noncommercial coursework, waits at least one second between requests, uses a 30-second timeout, stops on non-200/empty responses, verifies cached hashes, and remembers both successes and failures for the run. The default parser reads `data/raw/`; optional `python scripts/acquisition.py --chamber Senate --limit 1` retrieves the pinned public archive copy. House URLs point to official PDFs.

The static-file downloads are **not API calls**. Safe response headers are archived, including rate-limit headers if the server supplies them; no documented API limit is assumed. The Senate acquisition path uses previously collected source HTML rather than silently accepting an access agreement. Windows text checkout CRLF bytes and GitHub raw LF transport bytes have separate documented hashes; archived inputs are not rewritten.

In [2]:
import inspect
# These are the exact implementations used, displayed here for grading.
print("User-Agent:", USER_AGENT)
display(Code(inspect.getsource(CachedFetcher), language="python"))
plan = acquisition_plan()
print("Enumerated source documents:", len(plan))
assert len(plan) == 1098  # protects against a missing document inventory
sample = next(row for row in plan if "/senate/" in row["path"])
client = CachedFetcher()
body = client.fetch(sample["url"])
assert hashlib.sha256(body).hexdigest() == sample["sha256"]
assert client.fetch(sample["url"]) == body  # same URL reuses memory
assert all(event["from"] != "network" for event in client.events)
print("Offline repeat-run proof:", [event["from"] for event in client.events])
header_rows = []
for path in (ROOT / "data/raw/http_cache").glob("*.json"):
    meta = json.loads(path.read_text())
    rate_headers = {k:v for k,v in meta["headers"].items() if "ratelimit" in k.lower() or k.lower() == "retry-after"}
    header_rows.append({"status":meta["status"],"source":meta["url"],
                        "observed rate-limit headers":rate_headers or "none returned (static file)",
                        "retrieved UTC":meta["retrieved_at_utc"]})
display(pd.DataFrame(header_rows))


User-Agent: DSA405-P3-Coursework/1.0 (Sean Ryan; noncommercial academic acquisition)
class CachedFetcher:
    def __init__(self, cache_dir=None, interval=1.0, opener=None, clock=time.monotonic, sleep=time.sleep):
        if interval < 1.0:
            raise ValueError('Coursework rate limit must be at least one second.')
        self.cache_dir = Path(cache_dir or ROOT/'data/raw/http_cache')
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.interval, self.clock, self.sleep = interval, clock, sleep
        self.opener = opener or urllib.request.build_opener()
        self.last_request = None
        self.seen = {}
        self.events = []

    def fetch(self, url):
        # Both successful and failed attempts are memoized: no repeated resource in a run.
        if url in self.seen:
            result = self.seen[url]
            if isinstance(result, Exception): raise result
            self.events.append({'url':url,'from':'memory'})
            return result
      

### Demonstrate a loud failure
A maintenance page is valid HTML but not a transaction table. Continuing would silently produce an empty dataset. A second demonstration duplicates a summary key: `validate="one_to_one"` must reject it.

In [3]:
caught = failure_cases()
assert {case["caught"] for case in caught} == {"SourceStructureError", "MergeError"}
for case in caught:
    print(case["title"], "→", case["caught"] + ":", case["message"])
print("Both deliberately broken inputs were caught; no bad output was accepted.")


Missing HTML selector → SourceStructureError: Expected Senate transaction table matched nothing, or its headers changed.
Duplicate join key → MergeError: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
 ticker   month
   ABC 2025-01 ...
Both deliberately broken inputs were caught; no bad output was accepted.


## 2. Source parsing and comparable scope
**House:** P3 builds `data/processed/house_extracted.csv` from the 515 PDFs, pinned parser, ticker-cleaning code, and checked review decisions inside this repository. Its 7,667 rows are generated by this project rather than loaded from P2's finished CSV. A source/code signature and output checksum protect the committed cache. Run `python scripts/house.py --rebuild` to reparse every PDF in a new local workspace; the default build automatically regenerates a missing or outdated extract. The first full PDF rebuild takes several minutes. Earlier projects are credited as the origin of the preserved archive and code.

**Senate:** parse every one of the 583 archived electronic reports. Require the nine expected HTML headers and nine cells per transaction row. Missing tables, dates, amounts, files, and duplicate report/item identities stop the pipeline. The filing index keeps 97 deferred paper reports visible.

**Comparable population:** transaction dates in 2025, reported stock/fund records (`ST` for House; `Stock` for Senate), usable reported ticker text, and House resolver status `accepted`. Preserve all exclusions in a separate table. Preserve report-level identities and source URLs; keep amendments. Funds/ETFs can appear under these source labels. Ticker text is not a permanent security identifier, so overlap is descriptive rather than proof that securities are economically identical.

In [4]:
house_all = load_house(ROOT)
house_build = json.loads((ROOT / "data/processed/house_extraction.json").read_text())
assert house_build["rows"] == len(house_all)
print("P3 House extraction:", house_build["rows"], "rows from", house_build["parsed_filings"], "parsed filings")
print("All", house_build["recipe"]["pdf_count"], "bundled PDFs accounted for;", house_build["filings_without_rows"], "have no extracted rows")
print("Checked local review corrections:", house_build["review"]["changed_cells"], "value cells")
senate_all, senate_coverage = parse_archive(ROOT)
# Confirm the new strict parser reproduces the upstream archive's record identities and fields.
upstream = pd.read_csv(ROOT / "data/raw/senate/transactions_upstream.csv", dtype=str, keep_default_na=False)
upstream["row_key"] = upstream.report_id + ":" + upstream.item_number
audit_new = senate_all.assign(row_key=senate_all.filing_id + ":" + senate_all.row_number)
assert upstream.row_key.is_unique and audit_new.row_key.is_unique
assert set(upstream.row_key) == set(audit_new.row_key)
audit = audit_new.merge(upstream, on="row_key", validate="one_to_one", suffixes=("_new", "_upstream"))
for column in ["transaction_date", "ticker", "asset_name", "transaction_type"]:
    # Upstream converts the source placeholder -- to blank; the strict parser
    # retains it as evidence. Normalize ONLY that documented placeholder here.
    parsed_values = audit[column+"_new"].fillna("").replace({"--":""})
    assert parsed_values.eq(audit[column+"_upstream"].fillna("")).all(), column
print("Source ticker placeholder -- retained in", int(audit.ticker_new.eq("--").sum()), "rows; upstream stored these as blanks.")
house, house_excluded, house_counts = prepare_comparable(house_all, "House")
senate, senate_excluded, senate_counts = prepare_comparable(senate_all, "Senate")
display(pd.DataFrame([house_counts, senate_counts], index=["House", "Senate"]))
print("Senate filing coverage:", senate_coverage)
print("All 5,069 Senate parsed records reconcile to the upstream archived table.")


P3 House extraction: 7667 rows from 449 parsed filings
All 515 bundled PDFs accounted for; 66 have no extracted rows
Checked local review corrections: 2 value cells
Source ticker placeholder -- retained in 1625 rows; upstream stored these as blanks.
        input_rows  eligible_rows  ...  excluded_scope  excluded_ticker
House         7667           6394  ...             765                4
Senate        5069            518  ...             255               66

[2 rows x 5 columns]
Senate filing coverage: {'electronic_reports': 583, 'paper_reports_deferred': 97, 'index_rows': 680, 'parsed_rows': 5069}
All 5,069 Senate parsed records reconcile to the upstream archived table.


## 3. Ethics and A8 plan drift
Sean confirmed on October 23 that A8 named **House + Senate disclosures**. This keeps those sources. The exact A8 text was not available in this build; Sean should compare its wording before submission.

Implementation choices to review as potential drift:

- Use transaction year 2025 and a frozen source snapshot so another reader can reproduce the result offline.
- Read the previously collected Senate electronic HTML and expose deferred paper reports. New acquisition can verify the pinned source copy without submitting the Senate agreement.
- Aggregate each source to ticker-month before merging, because House and Senate legislators are different entities. Keep the transaction append separate from the required validated join.

**Guardrails:** no account login, paywall, CAPTCHA solving, circumvention, or submission of the Senate access agreement is performed by P3. Public reports still contain names and disclosed financial interests, so the assignment's “no personal data” language cannot honestly be described as literal absence of personal information. The intended scope is public official disclosure data for noncommercial coursework; omit unrelated private identifiers and make no credit, solicitation, or wrongdoing inference. Consult the instructor if their phrase excludes this public-disclosure topic.

The [Senate access notice](https://efdsearch.senate.gov/search/home/) lists restrictions on report use. The [House Clerk](https://disclosures-clerk.house.gov/FinancialDisclosure) is the official House source. This notebook records the scope and access path; it does not claim a legal determination or that a previous agreement grants new consent.

**Coverage:** The captured House archive has 515 PDFs, 449 with extracted rows and 66 without extracted rows. Senate has 583 electronic and 97 deferred paper reports across the index's 2021–2026 filing window. Their collection dates differ. Amount ranges are not exact amounts, and amendments may repeat economic events.

## 4. Join design — before writing the merge
**Do not merge transaction rows directly on ticker and month.** That would create a many-to-many multiplication when both chambers report several rows for the same ticker-month. Summarize each source first.

| Design item | Decision |
|---|---|
| Keys | reported ticker + calendar month, within 2025 |
| Left grain | one ticker-month in the House archive |
| Right grain | one ticker-month in the Senate electronic archive |
| Relationship | one-to-one after aggregation |
| Expected rows | **3,760**, independently counted as the union of key sets |
| Join type | outer: compare both sources while keeping unmatched keys |
| Unmatched values | retained with NA in the absent chamber; not filled with zero |

An inner join would discard 3,600 keys. Joining legislators across chambers does not answer this question. An append produces a common transaction table but does not compare the source summaries or prove join cardinality.

In [5]:
house_summary = aggregate(house, "house")
senate_summary = aggregate(senate, "senate")
design = join_design(house_summary, senate_summary)
display(pd.DataFrame([design]).T.rename(columns={0:"Declared before merge"}))
assert design["expected_rows"] == 3760  # protects the frozen snapshot's expected union size
assert len(house_summary) == 3570 and len(senate_summary) == 350


                                           Declared before merge
keys                                             [ticker, month]
left_grain     one row per ticker and calendar month in House...
right_grain    one row per ticker and calendar month in Senat...
relationship                                          one_to_one
join_type                                                  outer
expected_rows                                               3760
left_rows                                                   3570
right_rows                                                   350
both                                                         160
left_only                                                   3410
right_only                                                   190
unmatched      retained; opposite chamber values remain NA, n...
why            Compare reported ticker-month activity without...


### Run the declared merge
`validate=` rejects fan-out caused by duplicate summary keys. The union assertion in the next section independently rejects row loss from an accidental inner join.

In [6]:
joined = house_summary.merge(
    senate_summary,
    on=["ticker", "month"],
    how="outer",
    validate="one_to_one",
    indicator=True,
).sort_values(["ticker", "month"]).reset_index(drop=True)
display(joined.head(8))


  ticker    month  house_reports  ...  senate_purchases  senate_sales     _merge
0      A  2025-03            2.0  ...               NaN           NaN  left_only
1      A  2025-04            3.0  ...               NaN           NaN  left_only
2      A  2025-05            2.0  ...               NaN           NaN  left_only
3      A  2025-06            1.0  ...               NaN           NaN  left_only
4      A  2025-07            1.0  ...               NaN           NaN  left_only
5    AAL  2025-03            5.0  ...               NaN           NaN  left_only
6    AAL  2025-04            6.0  ...               NaN           NaN  left_only
7    AAL  2025-05            1.0  ...               NaN           NaN  left_only

[8 rows x 13 columns]


## 5. Verification suite
Each assertion protects a real assumption. These checks cover both unwanted row multiplication and lost unmatched records. The function is included in the repository and shown in full below.

In [7]:
display(Code(inspect.getsource(verify_join), language="python"))
checks = verify_join(joined, house_summary, senate_summary, design, house, senate)
for index, description in enumerate(checks, 1):
    print(f"PASS {index}: {description}")
assert verify_raw(ROOT) == 1105  # protects untouched original raw inputs
# Run the separate timeout, HTTP, cache, rate-limit, and deliberate join-failure tests too.
test_run = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(ROOT / "tests"), "-v"], capture_output=True, text=True)
print(test_run.stderr.strip())
assert test_run.returncode == 0, "Reliability tests failed"


def verify_join(joined,house,senate,design,house_transactions,senate_transactions):
    # Each assertion states the real data assumption it protects.
    assert len(joined)==design['expected_rows'],'Protects against fan-out or dropped keys'
    assert not joined.duplicated(KEYS).any(),'Protects against duplicated ticker-month summaries'
    assert set(map(tuple,joined[KEYS].to_numpy()))==(set(map(tuple,house[KEYS].to_numpy()))|set(map(tuple,senate[KEYS].to_numpy()))),'Protects against an accidental inner join dropping unmatched months'
    assert int(joined.house_reports.sum())==len(house_transactions),'Protects against lost or multiplied House report rows'
    assert int(joined.senate_reports.sum())==len(senate_transactions),'Protects against lost or multiplied Senate report rows'
    assert joined[KEYS].notna().all().all(),'Protects against missing merge keys matching each other'
    assert joined.month.str.fullmatch(r'2025-(0[1-9]|1[0-2])').all(),'Protects against joining different 

## 6. One thing that surprised me — evidence to review
The overlap is small: **160 of 3,760 ticker-month keys (4.26%) occur in both archives**. An inner join would remove **3,600 keys (95.74%)**. This makes the choice of an outer join material, rather than cosmetic.

This observation is supported by the archived records; the first-person reflection should be reviewed and rewritten by Sean as needed. It does not establish that the chambers' actual investments differ by that amount, because coverage, filings, amendments, and source formats differ.

In [8]:
coverage = joined["_merge"].value_counts().reindex(["left_only", "right_only", "both"])
display(coverage.rename(index={"left_only":"House only", "right_only":"Senate only", "both":"Both chambers"}).to_frame("Unique ticker-month keys"))
print(f"Overlap: {design['both'] / design['expected_rows']:.2%}")
print(f"Keys an inner join would lose: {design['left_only'] + design['right_only']:,}")
# A static, directly labeled count diagram; widths use a shared zero-based scale.
labels = [("House only",design["left_only"]),("Senate only",design["right_only"]),("Both chambers",design["both"])]
bars = []
for i, (label, count) in enumerate(labels):
    y = 65 + i*48
    bars.append(f'<text x="12" y="{y+14}" font-size="13">{label}</text><rect x="125" y="{y}" width="{count/design["expected_rows"]*390}" height="19" fill="#216453"/><text x="540" y="{y+14}" font-size="13">{count:,}</text>')
display(SVG('<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 620 240" width="620" role="img" aria-label="House only 3410; Senate only 190; both chambers 160 ticker-month keys"><rect width="620" height="240" fill="white"/><g font-family="Arial" fill="#162d36"><text x="12" y="25" font-size="17">2025 ticker-month keys by archive coverage</text>' + ''.join(bars) + '<text x="12" y="225" font-size="11">3,760 unique keys. Missing archive coverage is not an observed zero.</text></g></svg>'))


               Unique ticker-month keys
_merge                                 
House only                         3410
Senate only                         190
Both chambers                       160
Overlap: 4.26%
Keys an inner join would lose: 3,600
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 620 240" width="620" role="img" aria-label="House only 3410; Senate only 190; both chambers 160 ticker-month keys"><rect width="620" height="240" fill="white"/><g font-family="Arial" fill="#162d36"><text x="12" y="25" font-size="17">2025 ticker-month keys by archive coverage</text><text x="12" y="79" font-size="13">House only</text><rect x="125" y="65" width="353.6968085106383" height="19" fill="#216453"/><text x="540" y="79" font-size="13">3,410</text><text x="12" y="127" font-size="13">Senate only</text><rect x="125" y="113" width="19.707446808510635" height="19" fill="#216453"/><text x="540" y="127" font-size="13">190</text><text x="12" y="175" font-size="13">Both chambers</text><rec

## 7. Write reproducible outputs
The exclusion table accounts for every original row; source identities and URLs remain available. The interactive website is built from these same functions and output schemas.

In [9]:
output = ROOT / "data/processed"
output.mkdir(parents=True, exist_ok=True)
joined.to_csv(output / "combined_ticker_month.csv", index=False)
combined = pd.concat([house, senate.drop(columns="row_key",errors="ignore")], ignore_index=True)
assert combined.transaction_id.is_unique
combined.to_csv(output / "combined_transactions.csv", index=False)
pd.concat([house_excluded, senate_excluded], ignore_index=True).to_csv(output / "excluded_rows.csv", index=False)
print(f"Saved {len(joined):,} summary rows and {len(combined):,} comparable source rows.")
print("Raw inputs still verified:", verify_raw(ROOT))


Saved 3,760 summary rows and 6,912 comparable source rows.
Raw inputs still verified: 1105


## 8. Repair a real PDF character mapping

The PDF displays Filing Status: New correctly, but its embedded text mapping is broken. Recover the label and keep metadata separate from asset and ticker columns.

In [10]:
from unicode_example import main as demonstrate_unicode_repair
example=demonstrate_unicode_repair()
display(pd.DataFrame([{"Before":example["raw"],"After":example["cleaned"],"Saved field":example["destination_field"]}]))

{
  "filing_id": "20030924",
  "page": 1,
  "source": "data/raw/2025_pdfs/20030924.pdf",
  "original_url": "https://disclosures-clerk.house.gov/public_disc/ptr-pdfs/2025/20030924.pdf",
  "raw": "F\u028b\u028e\u028b\u0290\u0289 S\u0296\u0283\u0296\u0297\u0295: New",
  "cleaned": "Filing Status: New",
  "offset_decimal": 546,
  "offset_hex": "0x0222",
  "affected_range": "U+0283-U+029C",
  "ticker_already_correct": "ABBV",
  "destination_field": "filing_status",
  "destination_value": "New"
}
               Before               After    Saved field
0  Fʋʎʋʐʉ Sʖʃʖʗʕ: New  Filing Status: New  filing_status


## 9. Final portfolio snapshot

These bundled derived CSVs have a broader scope than the reproducible 2025 archive above. They are kept separate and their hashes are recorded.

In [11]:
from p4_synthesis import build_completion, docs, update_site
summary=json.loads((ROOT/"data/processed/validation.json").read_text())
snapshots=build_completion()
docs(summary,snapshots)
update_site()
display(pd.DataFrame(snapshots).T)
final_table=pd.read_csv(ROOT/"docs/data/portfolio-completion.csv",dtype=str,keep_default_na=False)
assert len(final_table)==31662
assert final_table.groupby("chamber").size().to_dict()=={"House":26732,"Senate":4930}
assert final_table.source_url.str.startswith(("https://","http://")).all()
print("Final snapshot rows:",len(final_table))
print("Dates requiring source review:",final_table.date_quality.ne("").sum())

Portfolio date anomalies preserved for source review: 4
                                                     file  ... transaction_date_max
House       data/portfolio/house_ptr_transactions_web.csv  ...           3031-04-30
Senate  data/portfolio/senate_ptr_transactions_electro...  ...           2026-07-31

[2 rows x 5 columns]
Final snapshot rows: 31662
Dates requiring source review: 4


## 10. Interpretation, missingness and AI reflection

The written report contains all nine required sections. Handwritten/image-only House records remain unresolved; 97 Senate paper reports are deferred. No parsed rows is not proof of no transactions. Amendments and household ownership limit claims about unique trades and personal spending.

In [12]:
print((ROOT/"AI_USE_LOG.md").read_text(encoding="utf-8"))
print((ROOT/"SELF_ASSESSMENT.md").read_text(encoding="utf-8"))

# AI use and reflection

ChatGPT helped develop code, explain methods, draft documentation, and identify checks. I directed the work, inspected source examples, and selected which methods to keep. This P4 draft and its suggested self-scores still need my final review before submission.

**My biggest lesson was not to let AI spend indefinitely going in circles on old ideas that I had already seen fail.** I had to interrupt that repetition, inspect the document's structure, and explore libraries and methods better suited to it. The productive change was treating the PDF as a positioned document: use table and rectangle geometry rather than repeatedly guessing from a loose stream of text.

The checked implementation uses PyMuPDF's page.find_tables(), fitz.Rect, and page.get_text(..., clip=rect). It is fair to describe the rectangle finder as a library-based PDF geometry approach. That does not prove why the model repeated suggestions internally, and a library is not automatically correct.

## 11. Handoff

Submit the repository URL, this notebook, reports/P4_REPORT.md, and SELF_ASSESSMENT.md. Review the first-person reflection and suggested self-scores before submission. See REQUIREMENTS.md for the evidence map.